# Extract mantle data

This notebook will extract mantle diagnostics from plate-model-driven G-ADOPT outputs, then save or append the resultant data to `training_data_global_mantle.csv`. This can then be used to train the models in later notebooks (`01*.ipynb`).

## Notebook options

These cells set some of the important variables and definitions used throughout the notebook.

In [ ]:
config_file = "config/.run_config.yml"

In [ ]:
from lib.load_params import get_params
from pathlib import Path

params = get_params(config_file, notebook="00d")

# =====================
# Plate model
# =====================

plate_model_name = params["plate_model"]["plate_model_name"]
use_provided_plate_model = params["plate_model"]["use_provided_plate_model"]

# =====================
# Filestructure
# =====================

# Parent data directory
parent_data_dir = Path(params["data_dir"])

# Directory for data derived from chosen reconstruction
recon_data_dir = parent_data_dir / plate_model_name

# Directory for extracted point data
output_dir = (
    recon_data_dir /
    "extracted_data" /
    f"{params["reference_feature"]}_{params["study_zone_buffer"]:.1f}_deg_buffer"
)

# CSV file with known deposits; columns:
# lon, lat, age (Ma), label, source
deposits_filename = parent_data_dir / "deposits" / params["deposits_filename"]

# If desired, categorise deposits according to location
# Should be a shapefile or GeoJSON containing polygons
# with a 'region' attribute
regions_filename = parent_data_dir / params["regions_filename"]

# Initialise filestructure
parent_data_dir.mkdir(parents=True, exist_ok=True)
recon_data_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

# =====================
# Run parameters
# =====================

# Number of processes to use
n_jobs = params["n_jobs"]

# Overwrite any existing output files
overwrite = params["overwrite_output"]

# Control verbosity level of logging output
verbose = params["verbose"]

# Timespan for analysis
min_time = params["timespan"]["min"]
max_time = params["timespan"]["max"]
times = range(min_time, max_time + 1)

# Number of unlabelled points to generate
num_unlabelled = params["num_unlabelled"]  # per timestep

# Random seed for reproducibility
random_seed = params["random_seed"]


## Notebook setup

Imports, definitions, etc.

### Imports

In [ ]:
import os
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately.tools import plate_isotherm_depth

from lib.assign_regions import assign_regions
from lib.calculate_convergence import run_calculate_convergence
from lib.check_files import (
    check_source_data,
    check_plate_model,
)
from lib.combine_point_data import combine_point_data
from lib.coregister_combined_point_data import run_coregister_combined_point_data
from lib.coregister_crustal_thickness import run_coregister_crustal_thickness
from lib.coregister_ocean_rasters import (
    extract_subducted_thickness,
    run_coregister_ocean_rasters,
)
from lib.create_study_area_polygons import run_create_study_area_polygons
from lib.erodep import calculate_erodep
from lib.generate_unlabelled_points import generate_unlabelled_points
from lib.misc import calculate_slab_flux, calculate_carbon
from lib.plate_models import get_plate_reconstruction
from lib.slab_dip import calculate_slab_dip
from lib.water import calculate_water_thickness

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning
warnings.simplefilter("ignore", UserWarning)

env: PYTHONWARNINGS=ignore::UserWarning


### Input and output files

If necessary, the plate model will be downloaded:

In [ ]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

43692kB [01:22, 528.17kB/s] 


In [ ]:
if use_extracted_data:
    print(f"Using extracted training data in {data_dir}")
else:
    data_dir = check_prepared_data("prepared_data", verbose=True)

# Directory for interpolated G-ADOPT output grids
data_dir = parent_data_dir / "g-adopt-outputs" / plate_model_name / "nc_output"

# Directory for plate model
plate_model_dir = recon_data_dir / "plate_model"

data_filename = os.path.join(data_dir, "training_data_global.csv")


output_deposits = os.path.join(output_dir, "deposits")
figures_dir = os.path.join(output_dir, "feature_selection")
os.makedirs(figures_dir, exist_ok=True)

pu_dir = os.path.join(output_dir, "PU")
os.makedirs(pu_dir, exist_ok=True)
pu_basename = os.path.join(pu_dir, "classifier")

importance_dir = os.path.join(pu_dir, "feature_importance")
os.makedirs(importance_dir, exist_ok=True)
importance_basename = os.path.join(importance_dir, "feature_importance")

svm_dir = os.path.join(output_dir, "SVM")
os.makedirs(svm_dir, exist_ok=True)
svm_basename = os.path.join(svm_dir, "classifier")

### 